# 01: Baseline Extraction
Extract embeddings from Qwen3-Omni falling back to 4-bit mode for free-tier GPUs.

In [ ]:
import os
import sys

# If running on a cloud Colab instance (Linux) rather than local Windows
if "COLAB_GPU" in os.environ or "COLAB_JUPYTER_IP" in os.environ or "google.colab" in sys.modules:
    print("Detected Google Colab Environment!")
    # Clone repository if it doesn't exist
    if not os.path.exists("OmniEmbed"):
        !git clone https://github.com/TinevimboMusingadi/OmniEmbed.git
    
    # Switch to directory and install pip dependencies
    %cd OmniEmbed
    !pip install -r requirements.txt
    !pip install -e .
else:
    print("Detected Local IDE / Non-Colab Environment")
    # Assuming the current working directory is the repo root or can access it
    get_ipython().system("pip install -e .")


Ensure HuggingFace token is set for gated checkpoints:

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Run the extraction script on Text and Image modalities via the CLI wrapper:
*Note: using 4bit flag drops exact accuracy slightly but enables T4 GPU runs*

In [ ]:
!python scripts/extract_embeddings.py --modality text --dataset coco --n_samples 50 --layer -1 --quantize 4bit
!python scripts/extract_embeddings.py --modality image --dataset coco --n_samples 50 --layer -1 --quantize 4bit

## 1. Inspecting the Raw Embeddings
Let's load the generated representations into numpy arrays to see their numeric spread.

In [ ]:
import numpy as np

text_embs = np.load('embeddings/raw/text_layer-1.npy')
image_embs = np.load('embeddings/raw/image_layer-1.npy')

print(f"Text Tensor Shape: {text_embs.shape} (Samples x Dimensions)")
print(f"Image Tensor Shape: {image_embs.shape} (Samples x Dimensions)")

print("\nFirst 5 dimensions of Text Embedding #0:")
print(text_embs[0, :5])

## 2. Basic Cross-Modal Search Operations
Using cosine similarity, we can rank the text embeddings that are geometrically closest to a given image embedding.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search_text_from_image(image_idx, top_k=5):
    # Extract query and reshape for sklearn
    query_emb = image_embs[image_idx].reshape(1, -1)
    
    # Compare the single image embedding against all text embeddings
    sim_scores = cosine_similarity(query_emb, text_embs)[0]
    
    # Sort indices by highest similarity descending
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    
    print(f"--- Search Results for Query Image #{image_idx} ---")
    for rank, idx in enumerate(top_indices, 1):
        print(f"Rank {rank}: Matched Text #{idx} (Cosine Similarity: {sim_scores[idx]:.4f})")
    
    return top_indices

# Perform the search using Image #0 as our query target
results = search_text_from_image(image_idx=0)

## 3. Visualisation Modules
Using the `omniembed.viz` package tools we implemented, let's visualize the modalities using UMAP limits and similarity heatmaps natively.

In [ ]:
from omniembed.viz import plot_modality_umap, plot_similarity_heatmap

embeddings_dict = {
    "Text": text_embs,
    "Image": image_embs
}

# Generate the UMAP projection map taking our multi-modal dictionary
plot_modality_umap(embeddings_dict, title="Un-Aligned Modality Distribution (UMAP)")

In [ ]:
# Examine the raw intersection geometry bounds taking a subset slice of the first 15 samples.
sim_matrix = cosine_similarity(text_embs[:15], image_embs[:15])

text_labels = [f"T{i}" for i in range(15)]
img_labels = [f"I{i}" for i in range(15)]

plot_similarity_heatmap(sim_matrix, text_labels, img_labels, title="Cross-Modal Raw Similarity Heatmap (First 15 Samples)")